# 01 · Data Loading
Downloads the Yale ED triage dataset from Kaggle, extracts triage-time features with R (the full file is too large for Colab RAM), and caches a parquet copy to Google Drive.

**Setup:** add your Kaggle credentials in Colab's 🔑 Secrets panel (`KAGGLE_USERNAME` + `KAGGLE_KEY`, or `KAGGLE_API_TOKEN`) with notebook access enabled.

In [ ]:
import os
from google.colab import userdata
for k in ["KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"]:
    try: os.environ[k] = userdata.get(k)
    except Exception: pass

In [ ]:
!pip install -q kagglehub
import kagglehub
path = kagglehub.dataset_download("maalona/hospital-triage-and-patient-history-data")
print(path)

In [ ]:
!ls -lh {path}

## Extract triage-time columns with R
Keeps only features known at triage (acuity, demographics, arrival info, vitals, chief complaint). This also removes most leakage risk.

In [ ]:
import glob
f = glob.glob(f"{path}/*")[0]
r_code = f'''
df <- get(load("{f}"))
writeLines(names(df), "/content/all_columns.txt")
keep <- grep("^(esi|age|gender|disposition|arrivalmode|arrivalhour_bin|arrivalday|previousdispo|triage_vital_|cc_)",
             names(df), value = TRUE)
write.csv(df[, keep], "/content/triage_subset.csv", row.names = FALSE)
cat("kept", length(keep), "of", ncol(df), "columns\\n")
'''
open("/content/extract.R", "w").write(r_code)

In [ ]:
!Rscript /content/extract.R

In [ ]:
import pandas as pd
df = pd.read_csv("/content/triage_subset.csv")
df.shape, df["disposition"].value_counts()

## Cache to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs("/content/drive/MyDrive/prae", exist_ok=True)
df.to_parquet("/content/drive/MyDrive/prae/triage_subset.parquet")